# A2LLM - train mini on MULTILINGUAL corpus (T4)

**default PC model - BEST value** | **T4 time: ~30 min**

Corpus: en (wikitext-103 + TinyStories) + hi + ur + bn = ~2.2 B tokens

1. Runtime → Change runtime type → **T4 GPU**
2. **Runtime → Run all**
3. Last cell downloads `a2llm-mini.pt`

PC band kar sakte ho - Colab cloud mein chalta hai. Tab open rakho.

In [ ]:
!rm -rf /content/a2llm
!git clone --depth 1 https://github.com/ayushrajdev9-cmyk/a2llm
%cd /content/a2llm


In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q numpy pytest pandas pyarrow

In [ ]:
# build multilingual corpus ~2.2 GB (~5-8 min)
!python scripts/prepare_data_large.py

In [ ]:
# TRAIN mini: 100000 steps, batch 128, ~30 min
!python scripts/train.py --preset mini --data data/pretrain_multilingual.txt --steps 100000 --batch-size 128 --out checkpoints/mini
!python scripts/generate.py checkpoints/mini/best.pt --prompt "ROMEO:" --max-new-tokens 80

In [ ]:
# Auto-publish: creates GitHub release + uploads checkpoint (no manual download, no PC needed)
import getpass, json, math, os, requests, torch
TOKEN = getpass.getpass("GitHub token (repo scope): ")
OWNER, REPO_NAME, TAG, ASSET, CKPT = "ayushrajdev9-cmyk", "a2llm", "v0.6.0", "a2llm-mini.pt", "checkpoints/mini/best.pt"
H = {"Authorization": f"token {TOKEN}", "Accept": "application/vnd.github+json"}
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
loss, ppl = float(ck["best_val_loss"]), math.exp(float(ck["best_val_loss"]))
notes = f"MODEL: mini\n**{ASSET}** — mini tier retrained on multilingual corpus (T4)\n- params: 1,056,096 | ctx: 128\n- best val_loss {loss:.4f}, ppl {ppl:.2f}"
r = requests.post(f"https://api.github.com/repos/{OWNER}/{REPO_NAME}/releases",
                  headers=H, json={"tag_name": TAG, "name": f"A2LLM {TAG} — mini", "body": notes})
if r.status_code == 422:  # tag exists -> update instead
    rid = requests.get(f"https://api.github.com/repos/{OWNER}/{REPO_NAME}/releases/tags/{TAG}", headers=H).json()["id"]
    r = requests.patch(f"https://api.github.com/repos/{OWNER}/{REPO_NAME}/releases/{rid}", headers=H, json={"body": notes})
r.raise_for_status(); rid = r.json()["id"]
with open(ASSET, "rb") as f:
    u = requests.post(f"https://uploads.github.com/repos/{OWNER}/{REPO_NAME}/releases/{rid}/assets?name={ASSET}",
                      headers={**H, "Content-Type": "application/octet-stream"}, data=f)
u.raise_for_status()
print(f"✅ RELEASED: https://github.com/{OWNER}/{REPO_NAME}/releases/tag/{TAG}")